# Automatic Differentiation

In that section we computed the gradient of a loss by hand, and it
took some care even for a model with two parameters. A deep network has
millions of parameters spread over dozens of layers, and every change to its
design would mean deriving and coding its gradient again. Can the computer
do this work for us, given nothing more than the code that computes the
loss?

The line model of that section shows what the work involves. Its
loss is the squared norm of the errors
$\mathbf{r} = \mathbf{X}\mathbf{w} - \mathbf{y}$ on three examples. To
differentiate the loss, we noticed that the errors depend linearly on the
parameters $\mathbf{w}$, with Jacobian $\mathbf{X}$, and wrote out the
backward pass $\nabla_{\mathbf{w}} L = \mathbf{X}^\top (2\mathbf{r})$
ourselves. At $\mathbf{w} = (1, 0)$ it gave $(2, 4)$. A deep network needs
the same work for every layer: derive its Jacobian, code the product with it,
and test the code. Finite differences avoid the derivation, but they cost
one evaluation of the loss per parameter, and in float32 they give only
three or four correct digits.

Deep learning frameworks do this work for us. While our code computes the
loss, the framework records each operation that it executes, such as a
matrix–vector product or a sum, along with the inputs of the operation. The
derivative of every such operation is built into the framework. Once the loss
is known, the framework walks through the record backward, from the loss to
the parameters, and applies the chain rule at each operation, as we did by
hand for the line model. This procedure is called *automatic
differentiation*, or *autograd* for short. It returns the derivative of the
computation up to floating-point rounding, at a small multiple of the cost
of computing the loss, whatever the number of parameters.

We first let the framework compute the gradient of the line model and
compare it with $(2, 4)$. We then control which operations are
recorded, differentiate programs with loops and branches, differentiate a
derivative, and finally see why frameworks walk the record backward rather
than forward.

In [1]:
import torch

## Computing Gradients

### The Forward Pass

We set up the line model with the framework's tensors. Each row
$(1, \textrm{input})$ of $\mathbf{X}$ describes one of the three examples,
$\mathbf{y}$ holds their labels, and the parameters start at
$\mathbf{w} = (1, 0)$. The function `loss` computes
$L(\mathbf{w}) = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2$. It is the forward
pass only: no derivative appears anywhere in the code.

In [2]:
X = torch.tensor([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])  # [1, input]
y = torch.tensor([1.0, 1.0, 0.0])
w = torch.tensor([1.0, 0.0])

def loss(w):
    return ((X @ w - y) ** 2).sum()

<!-- d2l:prose id=autograd-md-3 fw=mxnet, pytorch, tensorflow -->

Recording costs memory and time, so the framework records only the
operations whose results depend on tensors that we mark as needing a
gradient. We mark $\mathbf{w}$.
`requires_grad_(True)` marks `w`; the trailing underscore is PyTorch's
convention for a method that changes its tensor in place. From now on every
operation on `w` is recorded, and the gradient will appear in the attribute
`w.grad`, which is `None` until the first backward pass.

In [3]:
w.requires_grad_(True)

tensor([1., 0.], requires_grad=True)

We now run the forward pass.

In [4]:
L = loss(w)
L

tensor(1., grad_fn=<SumBackward0>)

The loss carries `grad_fn=<SumBackward0>`: it remembers the operation that
produced it, the last one in the record.

The record forms a *computational graph* (the figure). Its
nodes are the values computed along the way, from the parameters to the
loss, and an edge runs from each input of an operation to its result. The
framework records operations at the level at which it knows their
derivatives, so the loss of the line model becomes four operations: a
matrix–vector product, the subtraction of the labels, an elementwise square,
and a sum.

![The computational graph of $L = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2$ at $\mathbf{w} = (1, 0)$, with one node for $\mathbf{w}$ and for each value computed from it; the constants $\mathbf{X}$ and $\mathbf{y}$ are not drawn, and $\mathbf{r} \odot \mathbf{r}$ is the elementwise square. Each node shows its value (black) and the gradient of $L$ with respect to it (red). The forward pass (blue) runs from left to right; the backward pass (red) starts from $1$ at $L$ and applies the local rule of each operation, from right to left.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-comp-graph.svg)

### The Backward Pass

We now ask the framework for the gradient of the loss at $\mathbf{w}$ and
compare it with the $(2, 4)$ that we derived by hand.
<!-- d2l:prose id=autograd-md-8 fw=mxnet, pytorch -->

Calling `backward` on the loss runs the backward pass and stores the
gradient in `w.grad`.

In [5]:
L.backward()
w.grad

tensor([2., 4.])

The result is $(2, 4)$, the gradient
$2\mathbf{X}^\top(\mathbf{X}\mathbf{w} - \mathbf{y})$ that we derived by
hand in that section. There we multiplied by two factors,
$2\mathbf{r}$ and $\mathbf{X}^\top$. The backward pass of
the figure produces them from four local rules, one per
operation. It starts from the derivative $1$ of $L$ with respect to itself.
The sum copies this gradient to each of its three inputs, the square
multiplies it entry by entry by $2\mathbf{r}$, subtracting the constant
labels leaves it unchanged, and the matrix–vector product multiplies it by
$\mathbf{X}^\top$. Each rule uses only the local derivative of one operation
and values saved during the forward pass, here $\mathbf{r}$ and
$\mathbf{X}$. This is why training needs more memory than prediction: the
framework must keep such values until the backward pass has used them.

### Gradient Accumulation

Training computes a new gradient at every step, so it matters what happens
to the gradient of the previous step.
<!-- d2l:prose id=autograd-md-10 fw=mxnet, pytorch, tensorflow -->

We run the forward and the backward pass a second time, at the same
$\mathbf{w}$.

PyTorch adds the new gradient to the one already stored in `w.grad`, so
`w.grad` now holds twice the gradient. Accumulation is useful when a loss is
a sum of parts whose gradients are computed separately: calling `backward`
on each part adds up their gradients. A training step that needs a fresh
gradient must therefore first reset `w.grad`, for instance with
`w.grad.zero_()`. A forgotten reset is a common bug.

In [6]:
L = loss(w)
L.backward()
w.grad  # the new gradient was added to the old one

tensor([4., 8.])

### Vector-Valued Functions

So far the function that we differentiated returned a scalar. Often it
returns a vector, for instance the loss of each example in a minibatch,
before the losses are added up. Let $\mathbf{u}$ be a vector of $m$ values
computed from $n$ inputs $\mathbf{x}$. Its derivative is the $m \times n$
Jacobian $\mathbf{J}$ with entries $J_{ji} = \partial u_j / \partial x_i$,
whose row $j$ is the transposed gradient of $u_j$. The backward pass does not
compute $\mathbf{J}$. It starts from a vector $\mathbf{v}$ with one entry
per output and returns

$$\mathbf{J}^\top \mathbf{v} = \nabla_{\mathbf{x}} \left(\mathbf{v}^\top \mathbf{u}\right) = \sum_{j=1}^m v_j \nabla_{\mathbf{x}} u_j,$$

the gradient of a weighted sum of the outputs. This is the chain rule of
that section, $\nabla_{\mathbf{x}} z = \mathbf{J}^\top
\nabla_{\mathbf{u}} z$, for a scalar $z$ computed from $\mathbf{u}$ whose
gradient $\nabla_{\mathbf{u}} z$ is $\mathbf{v}$.

With $\mathbf{v} = \mathbf{1}$, the vector of all ones, the backward pass
returns the gradient of the sum of the outputs. For per-example losses this
is what we want: the gradient of the total loss. We differentiate the
per-example squared errors $r_i^2$ of the line model, whose sum is $L$.

PyTorch does not choose $\mathbf{v}$ for us: calling `backward` on a tensor
that is not a scalar raises an error unless we pass $\mathbf{v}$ as the
argument `gradient`. The name reflects the chain rule, since $\mathbf{v}$ is
the gradient arriving from the rest of a larger computation. Passing a
vector of ones is the same as calling `losses.sum().backward()`.

In [7]:
w.grad.zero_()
losses = (X @ w - y) ** 2  # one loss per example
losses.backward(gradient=torch.ones(3))  # = losses.sum().backward()
w.grad

tensor([2., 4.])

The result is $(2, 4)$, the gradient of the total loss.
Choosing instead $\mathbf{v} = \mathbf{e}_j$, the coordinate vector with a
$1$ in position $j$ and $0$ elsewhere (that section), makes
the backward pass return the gradient of $u_j$ alone, which is row $j$ of the
Jacobian. that section uses this to compute whole
Jacobians.

## Controlling the Graph

Frameworks record every operation on a marked tensor, and sometimes that is
more than we want. A value computed from the parameters may need to act as a
constant, and some computations, such as the update of the parameters,
should not be recorded at all.

### Detaching Values

A common case of the first kind is a target: a value that the output of a
model should move toward, but that the model itself computes. Reinforcement
learning trains models this way, with targets built from the model's own
predictions. The update should move the prediction toward the target. If
gradients also flowed into the target, the update could lower the loss by
moving the target toward the prediction instead, without improving the
prediction. We therefore *detach* the target: we keep its value but cut its
connection to the parameters, so that the backward pass treats it as a
constant.

A small example shows what detaching does to a gradient. Let $u = x^2$ and
$z = ux$ for each entry $x$ of $\mathbf{x} = (0, 1, 2, 3)$, so that
$z = x^3$. In the graph of the figure(a), $x$ reaches $z$
along two paths, through $u$ and directly. The chain rule adds their
contributions, $x \cdot 2x$ and $u$, to give $\partial z / \partial x =
3x^2$. If we replace $u$ by a detached copy $c$, which has the same value
but no history, the path through $u$ disappears
(the figure(b)), and $\partial z / \partial x = c = x^2$.

![The product $z = ux$ with $u = x^2$, (a) attached and (b) detached. Blue arrows are operations of the forward pass; red arrows show the backward pass, labeled with the local derivatives it multiplies by. Detaching $u$ creates a constant $c$ with the same value and no connection to $x$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-detach.svg)

`u.detach()` returns a new tensor with the values of `u` and no history.

In [8]:
x = torch.arange(4.0, requires_grad=True)
u = x * x
c = u.detach()  # values of u, no history
z = c * x
z.sum().backward()
x.grad

tensor([0., 1., 4., 9.])

The gradient is $(0, 1, 4, 9)$, which is $c = x^2$. Without the detach,
both paths carry gradient.

Detaching did not change `u`, which keeps its history, so we can reuse it.

In [9]:
x.grad.zero_()
z = u * x
z.sum().backward()
x.grad

tensor([ 0.,  3., 12., 27.])

The gradient is now $3x^2 = (0, 3, 12, 27)$, although $z$ has the same
value $(0, 1, 8, 27)$ in both computations. Detaching changes the
graph, not the values.

### Disabling Gradient Tracking

When we need only values, as when a trained model makes predictions,
recording wastes time and memory, and we switch it off. Updating the
parameters is another such computation: the step
$\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L(\mathbf{w})$ is not part
of any loss that we want to differentiate. We take one step of gradient
descent on the line model with the learning rate $\eta = 0.1$.

Operations inside a `torch.no_grad()` block are not recorded. The update must
happen inside such a block: while it records, PyTorch does not allow an
in-place change of a leaf tensor that requires gradients, such as `w`. The
gradient in `w.grad` is the $(2, 4)$ computed above.

In [10]:
with torch.no_grad():  # nothing in this block is recorded
    w -= 0.1 * w.grad
    L = loss(w)
w, L

(tensor([ 0.8000, -0.4000], requires_grad=True), tensor(0.4000))

The step moves $\mathbf{w}$ to $(0.8, -0.4)$ and lowers the loss from
$1$ to $0.4$, as we computed by hand in that section. Repeating
this step is how the models of the next chapter are trained.

The loss computed inside the block has no `grad_fn`: it is not attached to
any graph.

## Differentiating Programs

The loss of the line model is a fixed formula. Programs often are not: how
many times a loop runs, or which branch an `if` statement takes, can depend
on the data. Automatic differentiation handles such programs as they are,
and it can even differentiate a derivative that it computed.

### Control Flow

A language model reads sentences of different lengths, so its computation
runs for a different number of steps on each input. No single formula
describes such a program for all inputs, but none is needed. The framework
records the operations that actually run for the input at hand, and these
form an ordinary computational graph that the backward pass can walk. The
function `f` below doubles its input until the norm of the result reaches
1000, and then returns the result, multiplied by 100 unless its entries have
a positive sum.

In [11]:
def f(a):
    b = a * 2
    while torch.linalg.vector_norm(b) < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

For a scalar input $a$, the result is $f(a) = ka$ with a factor $k$ that
depends on $a$: a power of two, multiplied by $100$ when $a$ is negative.
Between the inputs at which $k$ changes, $f$ is linear, and its derivative
is $k = f(a)/a$. We compute both sides of this equation at $a = 0.3$ and at
$a = -0.75$.

In [12]:
for value in (0.3, -0.75):
    a = torch.tensor(value, requires_grad=True)
    fa = f(a)
    fa.backward()
    print(f'a = {value:5.2f}   f(a) / a = {fa.item() / value:6.0f}   '
          f'gradient = {a.grad.item():6.0f}')

a =  0.30   f(a) / a =   4096   gradient =   4096
a = -0.75   f(a) / a = 204800   gradient = 204800


The two inputs record different graphs (the figure).
For $a = 0.3$, the function doubles $12$ times and keeps the result, so
$k = 2^{12} = 4096$. For $a = -0.75$, it doubles $11$ times and multiplies
by $100$, so $k = 100 \cdot 2^{11} = 204{,}800$. In both cases the gradient
equals $f(a)/a$.

![The graphs recorded by `f` for two inputs. For $a = 0.3$, `f` doubles $a$ $12$ times and the first branch runs; for $a = -0.75$, it doubles $a$ $11$ times and the second branch runs. Each graph is linear in $a$, with its own slope.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-dynamic.svg)

Automatic differentiation differentiates the operations that ran, not the
function that the program represents. Where the number of doublings
changes, at $a = 1000/2^{12} \approx 0.24$ for instance, $f$ jumps and has
no derivative. Autograd still returns the slope of the piece that ran. For
`f`, the result is the derivative wherever the derivative exists, and a
number with no meaning at the jumps.

### Higher-Order Derivatives

Some methods need the derivative of a derivative: the curvature of a loss,
for instance, or the products of a vector with the matrix of second
derivatives, the *Hessian*, which some optimizers use. The backward pass is
a computation like any other, so the framework can record it and
differentiate it again. We take $g(x) = x^3$, with $g'(x) = 3x^2$ and
$g''(x) = 6x$, which at $x = 1$ equal $3$ and $6$.

`torch.autograd.grad` returns a gradient instead of storing it in `.grad`.
With `create_graph=True` it also records the backward pass, so the first
derivative is a tensor with a history, which we differentiate again.

In [13]:
x = torch.tensor(1.0, requires_grad=True)
dg = torch.autograd.grad(x ** 3, x, create_graph=True)[0]  # 3x²
d2g = torch.autograd.grad(dg, x)[0]                         # 6x
dg, d2g

(tensor(3., grad_fn=<MulBackward0>), tensor(6.))

Both values match the formulas.

## Forward and Reverse Mode

Every backward pass in this section started at an output and worked toward
the inputs. The chain rule does not prescribe this order. It multiplies the
local derivatives along each path through the graph, and the products can
be formed from either end. *Reverse mode*, the backward pass that we have
used, starts at an output. *Forward mode* starts at an input and carries
derivatives forward together with the values. Both compute the same
derivatives, at very different costs, and the costs decide which mode
trains networks.

### Rows and Columns of the Jacobian

One traversal of the graph, in either mode, is called a *sweep*; a reverse
sweep is a backward pass. Consider a function with $n$ inputs and $m$
outputs and its $m \times n$ Jacobian $\mathbf{J}$. Either sweep costs a
small multiple of one evaluation of the function, and each one fills one row
or one column of $\mathbf{J}$ (the figure).

As we saw for vector-valued functions, a reverse sweep starts from a vector
$\mathbf{v}$ at the outputs and returns $\mathbf{J}^\top \mathbf{v}$, the
transpose of the *vector–Jacobian product* $\mathbf{v}^\top \mathbf{J}$.
With $\mathbf{v} = \mathbf{e}_j$, this is row $j$ of $\mathbf{J}$: the
gradient of output $j$ with respect to all inputs. Filling the whole
Jacobian takes $m$ reverse sweeps, one per row.

A forward sweep starts from a direction $\mathbf{d}$ at the inputs and
returns the *Jacobian–vector product* $\mathbf{J}\mathbf{d}$, the rate at
which all outputs change as the inputs move along $\mathbf{d}$. With
$\mathbf{d} = \mathbf{e}_i$, this is column $i$: the derivatives of all
outputs with respect to input $i$. Filling the whole Jacobian takes $n$
forward sweeps, one per column.

![Two ways to fill the $m \times n$ Jacobian $\mathbf{J}$ of a function with $n$ inputs and $m$ outputs: (a) forward mode, $n$ sweeps in all; (b) reverse mode, $m$ sweeps in all.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-fwd-vs-rev.svg)

We fill the Jacobian of the line model's errors
$\mathbf{r}(\mathbf{w}) = \mathbf{X}\mathbf{w} - \mathbf{y}$ both ways. The
errors are $m = 3$ outputs of $n = 2$ inputs, and their Jacobian is
$\mathbf{X}$, so reverse mode needs three sweeps and forward mode two. The
rows of the $k \times k$ identity matrix `eye(k)` are the coordinate vectors
$\mathbf{e}_1, \ldots, \mathbf{e}_k$, so a loop over its rows supplies one
coordinate vector per sweep, and `stack` joins the results into a matrix
(that section).

`torch.func.vjp(errors, w0)` returns the value of `errors` at `w0` and a
function that maps $\mathbf{v}$ to $\mathbf{J}^\top \mathbf{v}$.
`torch.func.jvp(errors, (w0,), (d,))` returns the value and
$\mathbf{J}\mathbf{d}$, computed in one forward sweep.

In [14]:
def errors(w):
    return X @ w - y

w0 = torch.tensor([1.0, 0.0])
_, vjp = torch.func.vjp(errors, w0)
rows = torch.stack([vjp(v)[0] for v in torch.eye(3)])  # row by row
cols = torch.stack([torch.func.jvp(errors, (w0,), (d,))[1]
                    for d in torch.eye(2)], dim=1)  # column by column
print('reverse sweeps:', rows.tolist())
print('forward sweeps:', cols.tolist())

reverse sweeps: [[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]]
forward sweeps: [[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]]


`torch.func.jacrev` and `torch.func.jacfwd` assemble a Jacobian from such
reverse and forward sweeps.

### The Cost of a Gradient

For a loss, the comparison is lopsided. A loss has a single output, $m = 1$,
so its Jacobian is one row, the transposed gradient, with an entry for each
of the $n$ parameters. Reverse mode computes the whole gradient in one
sweep, at a small multiple of the cost of computing the loss, whatever the
number of parameters. Forward mode needs one sweep per parameter, $n$ in
all, as many evaluations of the loss as finite differences. For a network with
$n = 10^8$ parameters, one reverse sweep does the work of $10^8$ forward
sweeps. This is the general form of what we found for the line model in
that section, whose backward pass cost about as much as its
forward pass, and it is why deep learning frameworks train networks in
reverse mode. Backpropagation is reverse mode applied to a neural network.

Reverse mode needs more memory than forward mode, since the forward values
that the backward pass uses grow with the number of operations. Forward mode
carries one derivative along with each value, so it needs about twice the
memory of evaluating the function, however many operations there are. It is
the better choice
for functions with few inputs and many outputs, and the two modes combine
well: forward mode applied to a gradient computed in reverse mode yields a
Hessian–vector product at the cost of a few gradients.

Both modes are older than deep learning. Forward mode goes back to
@Wengert.1964, and reverse mode was developed by
@Linnainmaa.1970 . that section
derives both modes in detail, including their costs in time and memory.

We can now compute the gradient of any loss that our code computes, control
which parts of the computation the gradient flows through, and take a
gradient step. We have treated the backward pass through a network as a
black box; that section opens it for a network with one hidden
layer. A gradient tells us how to lower the loss on the data at hand.
Whether a lower loss reflects a pattern that will hold on new data, or only
noise in the data we have, is a question of probability and statistics,
the subject of the next section.

## Summary

Automatic differentiation computes the gradient of a loss from the code
that computes the loss, without any derivative written by hand. The framework records the
operations of the forward pass as a computational graph and then applies the
chain rule backward through the graph, one operation at a time; for the line
model this reproduces the gradient $(2, 4)$ that we derived by hand.
Detaching a value turns it into a constant for the backward pass, and
disabling recording keeps computations such as parameter updates out of the
graph. Since the graph records what actually ran, loops and branches need no
special treatment, and the backward pass can itself be differentiated.
Reverse mode computes a whole gradient in one sweep, whatever the number of
parameters, which is why it trains the networks of this book.

## Exercises

1. [code] **Detached intermediates.** Let `u = x * x` and `c = u.detach()`,
   and consider `z = c * c`, `z = u * c`, and `z = u * u`. For each `z`,
   recompute `u` and `c`, reset `x.grad`, and call `backward` on the sum of
   the entries of `z`. Before running any code, predict the outcome of each
   call, including the resulting `x.grad`. Check your predictions, and
   explain the ratio between the gradients for `z = u * c` and `z = u * u`.

2. [code] **Manual parameter updates.** Perform the gradient step of this
   section as `w = w - 0.1 * w.grad`, outside `torch.no_grad()`, and inspect
   `w.requires_grad`, `w.grad_fn`, `w.is_leaf`, and `w.grad`. Explain what
   the update has done to the graph, predict what a second forward and
   backward pass leaves in `w.grad`, and check.

3. [code] **From scalar to vector input.** Replace the scalar input `a` of
   the function `f` defined in that section by a
   random vector of length 5, and differentiate the sum of the entries of
   `fa = f(a)` with respect to `a`.
    1. Express `f(a)` in terms of `a`, and give the Jacobian of `f` at `a`
       and the gradient of the sum of the entries of `fa`.
    1. Compute this gradient with automatic differentiation and check
       whether it still equals `fa / a` elementwise.
    1. Describe the set of inputs `a` at which `f` jumps. Which part of
       this set has no counterpart for a scalar input?

1. [code] **Programs versus functions.** Automatic differentiation
   differentiates the operations that a program executes, not the function
   that the program computes.
    1. The functions $|x|$ and $\sqrt{x^2}$ are equal. Compute the
       derivative of each at $x = 0$ by automatic differentiation, using the
       framework's absolute value and square root. Explain both results from
       the local derivatives of the operations involved. Neither function
       has a derivative at $0$; which of the two results does more harm in
       training, and why?
    1. Write a program that returns $x^2$ for every input $x$ but whose
       automatic derivative at $x = 0$ is $1$ instead of $0$. Explain why
       automatic differentiation cannot detect the difference.

1. [code] **Manual tracing.** Let $f(x, y, z) = (x + y) \cdot z$ with
   $x = -2$, $y = 5$, and $z = -4$.
    1. Draw the computational graph of $f$ with one node per value,
       compute the forward pass by hand, and trace
       $\partial f/\partial x$, $\partial f/\partial y$, and
       $\partial f/\partial z$ backward from $f$ through the multiplication
       and addition nodes (reverse mode).
    1. Compute the same three derivatives in forward mode, propagating the
       derivative with respect to one input at a time from the inputs to
       $f$. Compare the number of sweeps each mode needs and the
       intermediate values each must keep in memory.
    1. Verify every value using automatic differentiation.

    *Adapted from Stanford CS231n,
    ["An Exercise in Backpropagation"](https://cs231n.stanford.edu/slides/2026/section_2_backprop.pdf).*

1. **Cost of second derivatives.** ● The counting argument in
   that section puts the cost of the gradient of a
   scalar loss with $n$ parameters at one reverse sweep. Use the same argument to
   estimate, in units of function evaluations, the cost of the full Hessian
   (the $n \times n$ matrix of second derivatives) and the cost of a single
   Hessian–vector product. Explain the difference between the two costs.

[Discussions](https://d2l.discourse.group/t/35)